In [ ]:
import pandas as pd

# Load Data

In [ ]:
# data = pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\tonghopdata.csv')

data = pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\GSE183635_TEP_Count_Matrix.csv')
# name_gene =  pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\183635\Human_CH38p13annot.csv', low_memory=False    )


In [ ]:
data

,Unnamed: 0,1-Vumc-HD-101-TR922,2-Vumc-HD-103-TR923,3-Vumc-HD-108-TR924,4-Vumc-HD-127-TR925,5-Vumc-HD-130-TR926,6-Vumc-HD-142-TR927,7-Vumc-HD-143-TR928,8-Vumc-HD-144-TR929,9-Vumc-HD-145-TR930,...,countMatrix.3502-NKI-4854-SARC-TR3080,countMatrix.3503-NKI-4707-SARC-TR3079,countMatrix.3504-NKI-4941-SARC-TR3084,countMatrix.3507-NKI-6348-SARC-TR3134,countMatrix.3508-NKI-5275-SARC-TR3099,countMatrix.3512-NKI-5148-SARC-TR3129,countMatrix.320-10,countMatrix.327-34,countMatrix.376-P-PH-001-013-TR1400,countMatrix.506-NKI-NSCLC-107-270
0,ENSG00000000419,5,79,58,128,93,12,0,0,18,...,3,0,1,2,9,16,22,21,0,1
1,ENSG00000000460,8,10,21,14,6,8,25,42,5,...,0,3,0,2,63,3,3,0,1,0
2,ENSG00000000938,24,336,334,59,120,22,31,13,19,...,8,45,1,7,163,90,46,75,14,8
3,ENSG00000001036,38,46,15,36,25,43,27,27,59,...,1,3,4,15,3,6,10,8,3,0
4,ENSG00000001461,72,17,11,9,18,28,5,6,49,...,1,1,0,0,12,2,6,6,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5435,ENSG00000271043,349,133,99,87,171,260,232,204,188,...,0,0,0,5,0,2,18,31,22,23
5436,ENSG00000272053,980,658,294,265,818,2066,1579,1161,2003,...,18,17,17,107,41,52,78,203,12,16
5437,ENSG00000272168,51,20,11,31,65,49,56,25,131,...,0,0,0,26,0,1,15,6,10,0
5438,ENSG00000272369,29,41,24,3,51,33,26,52,35,...,0,3,1,8,2,1,0,3,2,1


In [ ]:
import numpy as np

# Minimal TMM normalization adapted from edgeR logic
# Returns CPM and factors scaled so geometric mean of factors is 1.
def compute_tmm_cpm(counts_df, trim_M=0.3, trim_A=0.05, ref_sample=None):
    counts = counts_df.copy()
    lib_sizes = counts.sum(axis=0)

    # Pick reference sample: closest library size to median if none provided
    if ref_sample is None:
        ref_sample = (lib_sizes - lib_sizes.median()).abs().idxmin()
    ref_counts = counts[ref_sample]
    lib_ref = lib_sizes[ref_sample]

    factors = {}

    def tmm_factor(test_counts, lib_test):
        mask = (test_counts > 0) & (ref_counts > 0)
        if mask.sum() == 0:
            return 1.0

        test = test_counts[mask]
        ref = ref_counts[mask]

        logR = np.log2((test / lib_test) / (ref / lib_ref))
        logA = 0.5 * np.log2((test / lib_test) * (ref / lib_ref))
        weights = 1.0 / ((1.0 / test) + (1.0 / ref))

        # Trim extremes on M and A
        m_low, m_high = np.quantile(logR, [trim_M / 2, 1 - trim_M / 2])
        a_low, a_high = np.quantile(logA, [trim_A / 2, 1 - trim_A / 2])
        keep = (logR >= m_low) & (logR <= m_high) & (logA >= a_low) & (logA <= a_high)

        if keep.sum() == 0:
            return 1.0

        weighted_M = np.average(logR[keep], weights=weights[keep])
        return 2 ** weighted_M

    for col in counts.columns:
        factors[col] = tmm_factor(counts[col], lib_sizes[col])

    factors = pd.Series(factors)
    factors = factors / np.exp(np.log(factors).mean())  # rescale to geometric mean 1
    effective_lib_sizes = lib_sizes * factors

    cpm = counts.divide(effective_lib_sizes, axis=1) * 1_000_000
    return cpm, factors, effective_lib_sizes

In [ ]:
# Apply TMM normalization → CPM → log2(CPM+1)
count_cols = data.columns.drop('Unnamed: 0')
counts_numeric = data[count_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# TMM factors trimmed at 30% for M and 5% for A (edgeR defaults)
tmm_cpm, tmm_factors, tmm_libsizes = compute_tmm_cpm(
    counts_numeric,
    trim_M=0.3,
    trim_A=0.05,
)

tmm_log2 = np.log2(tmm_cpm + 1)

# Reattach gene column and use the normalized values for downstream steps
tmm_log2.insert(0, 'Unnamed: 0', data['Unnamed: 0'])
raw_counts = data.copy()
data = tmm_log2

tmm_log2.head()

In [ ]:
data

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
# Transpose so samples become rows
data_transposed = data.set_index('Unnamed: 0').T
data_transposed

In [ ]:
# data_transposed.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\otherdata can be merged\raw.csv', index=True )

In [ ]:
data_transposed['cancer_type'] = (
    data_transposed['cancer_type']
    .replace('Other/Unknown', 'Asymptomatic_Controls')
)


In [ ]:
data_transposed['cancer_type'].value_counts()

In [ ]:
# Extract the names of the remaining 338
remaining_unknowns = [c for c in data.columns if categorize_tumor_type(c) == 'Other/Unknown']

# Look for patterns

print(pd.Series(remaining_unknowns).head(44))

In [ ]:
data_transposed = data_transposed[
    data_transposed['cancer_type'] != 'Symptomatic_Control'
]

In [ ]:
data_transposed.reset_index(drop=True, inplace=True)


In [ ]:
data_transposed

In [ ]:
# 4) Filter low-expression genes (expressed in ≥10% of samples)
min_samples = len(data_transposed) * 0.01
gene_cols = data_transposed.columns.drop('cancer_type')
expressed_genes_mask = (data_transposed[gene_cols] > 0).sum(axis=0) >= min_samples
data_filtered = data_transposed.loc[:, expressed_genes_mask.index[expressed_genes_mask]]
data_filtered['cancer_type'] = data_transposed['cancer_type']


In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
data_normalized = data_filtered.copy()

In [ ]:
# # # 5) MinMax normalize gene expression values
# # data_normalized = data_filtered.copy()

# # Get gene columns (exclude cancer_type)
# gene_cols = data_normalized.columns.drop('cancer_type')

# # Initialize MinMaxScaler
# scaler = MinMaxScaler()

# # Fit and transform only the gene expression columns
# data_normalized[gene_cols] = scaler.fit_transform(data_filtered[gene_cols])

# # Keep cancer_type column
# data_normalized['cancer_type'] = data_filtered['cancer_type']


In [ ]:
data_normalized

In [ ]:
# 6) Split features/labels
X = data_normalized.drop(columns=['cancer_type'])
y = data_normalized['cancer_type']


In [ ]:
X

In [ ]:
# y.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Multi\data\y_multi_ori.csv', index=False)

In [ ]:
# X.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Multi\data\normalized_X_multi_log.csv', index=False)

In [ ]:
# X.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\X_normalized_binary_log.csv', index=False)

In [ ]:
y = y.apply(lambda x: 1 if x != 'Asymptomatic_Controls' and x != 'Symptomatic_Control' else 0)

In [ ]:
y.value_counts()

In [ ]:
# y.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\y_normalized_binary.csv', index=False)

# embedded RF with stratify k fold

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Set random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# X = pd.read_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\X_normalized_binary_log.csv", low_memory=False)
# y = pd.read_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\y_normalized_binary.csv", low_memory=False).values.ravel()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
# X_train.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_train.csv", index= False)
# X_test.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_test.csv", index= False)
# y_train.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\y_train.csv", index= False)
# y_test.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\y_test.csv", index= False)


In [ ]:

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
gene_names = X_train.columns.values

In [ ]:
print("  This may take a few minutes...")

# Parameter grid for RandomForest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5]
}

# RandomForest classifier
rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)

# Stratified K-Fold
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Grid search
grid_search_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=cv_inner,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_rf.fit(X_scaled, y_train)

# Get feature importances
best_rf = grid_search_rf.best_estimator_
feature_importances = best_rf.feature_importances_

### Ranking all genes by RandomForest importance...

In [ ]:
# Create dataframe with ALL genes and their importance scores
rf_df = pd.DataFrame({
    'gene': gene_names,
    'importance': feature_importances
})
rf_df = rf_df.sort_values('importance', ascending=False)

In [ ]:
rf_df

In [ ]:
rf_df.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\feature_importance.csv', index=False)

In [ ]:
all_features = (
    rf_df.sort_values('importance', ascending=False)
    .head(5440)['gene']
    .tolist()
)

In [ ]:
X_train[all_features]

In [ ]:
X_test[all_features]

In [ ]:
X_train[all_features].to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_train.csv', index=False)
X_test[all_features].to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_test.csv', index=False)

In [ ]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(6, 6))
# plt.barh(importance_df['Feature'], importance_df['Importance'], color='blue')
# plt.xlabel('Importance Score')
# plt.ylabel('Feature')
# plt.title('Feature Importance Scores from Random Forest')
# plt.gca().invert_yaxis()  # To display the highest importance at the top
# plt.show()

# Start ML


In [153]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [154]:
X_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\X_train_rF_DEG_train_lung.csv",low_memory = False)
X_test = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\X_test_rF_DEG_train_lung.csv",low_memory = False)

y_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\y_train_rF_DEG_train.csv",low_memory = False)
y_test = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\y_test_rF_DEG_train.csv",low_memory = False)

In [155]:
import pandas as pd
import mygene

# 2. Extract the Ensembl IDs
# We filter for columns starting with 'ENSG' to avoid messing up index columns like 'Unnamed: 0'
ensembl_ids = [col for col in X_train.columns if col.startswith('ENSG')]

# (Optional) If your IDs have decimals like 'ENSG00000096060.1', you need to strip them:
# ensembl_ids = [col.split('.')[0] for col in ensembl_ids]

# 3. Query the MyGene Database
print("Querying MyGene.info for gene symbols...")
mg = mygene.MyGeneInfo()
results = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

# 4. Create a mapping dictionary { 'ENSG_ID' : 'Gene_Symbol' }
id_to_symbol = {}
for res in results:
    query_id = res['query']
    
    # If a gene symbol is found, use it. Otherwise, keep the original Ensembl ID.
    if 'symbol' in res:
        id_to_symbol[query_id] = res['symbol']
    else:
        id_to_symbol[query_id] = query_id

# 5. Rename the columns in both DataFrames
X_train.rename(columns=id_to_symbol, inplace=True)
X_test.rename(columns=id_to_symbol, inplace=True)

print("Conversion complete!")
print("Preview of new columns in X_train:")
print(X_train.head())
print("Preview of new columns in X_test:")
print(X_test.head())

Querying MyGene.info for gene symbols...


5 input query terms found no hit:	['ENSG00000197459', 'ENSG00000267243', 'ENSG00000136653', 'ENSG00000163386', 'ENSG00000148362']


Conversion complete!
Preview of new columns in X_train:
   WDR11-DT    MAGOHB   ZNF542P    IFITM3   HSD17B3     NOMO1   ITGB3BP  \
0  5.590922  4.920421  4.513914  6.808245  3.643703  5.229449  5.296728   
1  7.111388  6.283755  6.129917  4.970461  4.523772  7.441317  7.797965   
2  4.176511  3.782071  2.764579  7.538766  2.764579  3.782071  4.889779   
3  5.002632  4.644839  3.753215  5.804742  2.764579  4.836138  6.369995   
4  6.255441  5.619431  4.050363  7.027353  3.522978  4.234493  6.180337   

   LINC01088    COL6A3  ENSG00000260661  ...   CENATAC     LUC7L      DPP4  \
0   3.774722  5.391791         2.764579  ...  3.485998  4.452266  3.485998   
1   4.749558  3.944942         3.610526  ...  2.764579  3.944942  3.192072   
2   2.764579  4.889779         4.176511  ...  5.344660  6.186679  2.764579   
3   3.753215  4.137808         2.764579  ...  6.427103  7.613563  4.137808   
4   4.050363  5.435232         2.764579  ...  5.367917  5.435232  2.764579   

        LTF      PRCP   

In [131]:
X_test

,WDR11-DT,MAGOHB,ZNF542P,IFITM3,HSD17B3,NOMO1,ITGB3BP,LINC01088,COL6A3,ENSG00000260661,...,CENATAC,LUC7L,DPP4,LTF,PRCP,IL1R2,RRP12,ASGR2,RPS4Y1,TNFRSF10C
0,6.451547,5.348242,5.063048,7.340270,4.175718,5.665411,8.002006,4.625584,5.027229,3.306779,...,4.346157,4.521600,3.711639,2.830723,4.138130,2.830723,2.830723,2.830723,5.742559,2.830723
1,6.630307,5.769564,5.769564,3.921878,4.450995,5.844078,6.824695,4.551511,5.373541,3.728479,...,4.886915,5.026358,3.470567,7.424813,5.026358,3.921878,6.730802,4.644246,2.830723,3.921878
2,4.806340,4.468396,2.830723,7.664287,3.783802,4.345067,4.579964,3.783802,6.068926,2.830723,...,6.057216,6.159271,2.830723,2.830723,4.972535,3.988061,3.783802,6.068926,7.984931,4.579964
3,6.948542,6.390590,5.743158,5.220952,3.531932,6.448757,7.818158,4.996651,4.023021,4.564307,...,3.751771,3.531932,2.830723,4.535230,2.830723,3.328968,4.023021,2.830723,2.830723,3.924027
4,7.263197,6.852681,6.983045,5.905861,4.972501,6.667009,7.623942,4.838434,3.564025,4.543893,...,2.830723,3.401148,2.830723,2.830723,4.221185,3.161482,2.830723,3.401148,5.322133,3.161482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,7.793642,7.068577,6.270715,4.704627,5.540731,6.599390,9.150536,4.909011,5.282180,4.881746,...,2.830723,2.830723,2.830723,2.830723,3.223397,2.830723,2.830723,3.223397,3.223397,2.830723
179,6.029134,4.743265,5.561667,4.795813,3.522366,4.875604,7.151842,3.920887,5.369384,3.897971,...,2.830723,3.441971,2.830723,2.830723,3.063233,2.830723,2.830723,2.830723,2.830723,2.830723
180,6.585824,4.597360,5.647809,6.105820,4.280496,6.059986,8.286562,4.396034,6.193255,4.339747,...,4.848015,5.493568,2.830723,2.830723,2.830723,2.830723,4.449686,3.610574,6.044371,4.217875
181,4.978482,4.240479,4.631622,7.745341,2.830723,4.926833,6.610647,3.773898,5.122217,3.605274,...,3.913699,2.830723,2.830723,2.830723,3.913699,2.830723,2.830723,2.830723,5.166854,3.381660


In [132]:
# X_train = X_train.iloc[:,:300]
# 0.00055

In [133]:
# X_test = X_test.iloc[:,:300]

In [134]:
X_train = X_train.iloc[:,:410]
X_test = X_test.iloc[:,:410]
# cut at >= 0.00080 point

In [135]:
from sklearn.model_selection import StratifiedKFold

In [136]:
from xgboost import XGBClassifier

In [137]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
x_train_sc = scale.fit_transform(X_train)
x_test_sc = scale.transform(X_test)

In [138]:
list_feature = list(X_train.columns)

In [139]:
ranks = []
for i in range(len(list_feature)):
    arr = []
    for j in range(i+1):
        arr.append(list_feature[j])
    ranks.append(arr)

In [140]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [141]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

pipe = Pipeline([
    ("smote", SMOTE(k_neighbors=3, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced",  
        probability=True,
        random_state = 42
    ))
])

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [142]:
from sklearn.metrics import confusion_matrix,roc_auc_score
def cfm(y_test,y_pred,y_prob):
    auc = roc_auc_score(y_test,y_prob)
    tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
    acc = accuracy_score(y_test,y_pred)
    recall = tp/(tp+fn)
    precision = tp/(tp+fp)
    specificity = tn/(tn+fp)
    sensitivity = tp / (tp+fn)
    f1_score = round((2*recall*precision)/(recall+precision),3)
    return np.array([acc,recall,precision,specificity,sensitivity,f1_score,auc])

In [143]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer # Skopt uses specialized spaces
from sklearn.metrics import accuracy_score
import numpy as np

# Grid search

In [144]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, confusion_matrix, accuracy_score, roc_curve

# --- 1. Custom Scorers (Same as previous models) ---
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp)
    return 0.0

# Safe definition for Windows Multiprocessing
specificity_scorer = make_scorer(recall_score, pos_label=0)

def balanced_accuracy_medical(y_true, y_pred):
    sn = recall_score(y_true, y_pred, zero_division=0)
    sp = specificity_score(y_true, y_pred)
    return (sn + sp) / 2

scoring_metrics = {
    'ACC': 'accuracy',
    'Recall': 'recall',
    'Precision': 'precision',
    'Sp': specificity_scorer,
    'Sn': 'recall',
    'f1_Score': 'f1',
    'AUC': 'roc_auc',
    'BA': make_scorer(balanced_accuracy_medical) 
}

# --- 2. Main execution block ---
if __name__ == "__main__":
    
    # Initialize LDA
    model = LinearDiscriminantAnalysis(priors=[0.5, 0.5])

    # --- Explain the Parameters ---
    # LDA throws errors if you mix certain solvers with shrinkage.
    # Using a list of dictionaries ensures GridSearch only tests valid combinations.
    params = [
        # 'svd' is the default solver. It doesn't use shrinkage. Good for data with many features.
        {'solver': ['svd']},
        # 'lsqr' and 'eigen' can use 'shrinkage', which acts like regularization (similar to alpha/C).
        # This helps immensely if your overlapping genes in PCA are causing multicollinearity.
        {'solver': ['lsqr', 'eigen'], 'shrinkage': ['auto', 0.1, 0.5, 0.9]}
    ]

    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=params, 
        cv=5, 
        verbose=1, 
        scoring=scoring_metrics,
        refit='BA', # Selects best solver based on Balanced Accuracy
        return_train_score=True, 
        n_jobs=-1
    )

    result = []
    result2 = []

    # --- 3. The Feature Ranking Loop ---
    # Ensure ranks, x_train_sc, y_train, x_test_sc, y_test, and cfm are defined
    for i, rank in enumerate(ranks[10:], start=10):
        X_current = x_train_sc[:, :i+1]
        y_current = y_train['cancer_type']

        # Fit the grid search
        grid_search.fit(X_current, y_current)
        
        best_idx = grid_search.best_index_
        cv_res = grid_search.cv_results_
        best_model = grid_search.best_estimator_
        
        def fmt(metric):
            return f"{cv_res[f'mean_test_{metric}'][best_idx] * 100:.2f}%"

        # --- THRESHOLD LOCK LOGIC ---
        # 1. Get probabilities for the training set
        y_prob_train = best_model.predict_proba(X_current)[:, 1]
        
        # 2. Use ROC curve to find the optimal threshold to balance Sn and Sp
        fpr_tr, tpr_tr, thresholds_tr = roc_curve(y_current, y_prob_train)
        best_threshold = thresholds_tr[np.argmax(tpr_tr - fpr_tr)]

        # Append Training Results
        result.append([
            rank,
            fmt('ACC'),
            fmt('Recall'),
            fmt('Precision'),
            fmt('Sp'),
            fmt('Sn'), 
            fmt('f1_Score'),
            fmt('AUC'),
            grid_search.best_params_
        ])

        df_result = pd.DataFrame(result)
        df_result.columns = ['Gene','ACC','Recall','Precision','Sp','Sn','f1_Score','AUC','Parameter']
        df_result.to_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_on_training_6_6.csv", index=False)

        # --- TEST SET PREDICTION WITH LOCKED THRESHOLD ---
        y_prob_test = best_model.predict_proba(x_test_sc[:, :i+1])[:, 1]
        
        # Apply the optimized threshold instead of 0.5
        y_pred_test = (y_prob_test >= best_threshold).astype(int)
        
        # Evaluate using your custom cfm function
        evaluate = cfm(y_test, y_pred_test, y_prob_test)
        
        result2.append([
            rank,
            '%.2f'%(evaluate[0]*100)+'%', # ACC
            '%.2f'%(evaluate[1]*100)+'%', # Recall/Sn
            '%.2f'%(evaluate[2]*100)+'%', # Precision
            '%.2f'%(evaluate[3]*100)+'%', # Sp
            '%.2f'%(evaluate[4]*100)+'%', # Sn
            '%.2f'%(evaluate[5]*100)+'%', # f1
            '%.2f'%(evaluate[6]*100)+'%', # AUC
            grid_search.best_params_
        ])

        df_result2 = pd.DataFrame(result2)
        df_result2.columns = ['Gene','ACC','Recall','Precision','Sp','Sn','f1_Score','AUC','Parameter']
        df_result2.to_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_DEG_on_training(test_selected)_6_6.csv", index=False)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 9 candidates

In [145]:
# df_result = pd.DataFrame(result)
# df_result.columns = ['Gene','ACC','Recall','Precision','Sp','Sn','f1_Score','AUC','Parameter']
df_result

,Gene,ACC,Recall,Precision,Sp,Sn,f1_Score,AUC,Parameter
0,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",82.33%,79.15%,88.72%,86.57%,79.15%,83.65%,89.72%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"
1,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",82.33%,79.38%,88.53%,86.25%,79.38%,83.70%,89.56%,"{'shrinkage': 0.9, 'solver': 'lsqr'}"
2,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",82.33%,78.90%,88.95%,86.89%,78.90%,83.62%,90.01%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"
3,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",82.05%,78.19%,89.14%,87.21%,78.19%,83.27%,89.95%,"{'shrinkage': 'auto', 'solver': 'lsqr'}"
4,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",81.92%,76.50%,90.37%,89.13%,76.50%,82.86%,91.32%,"{'shrinkage': 'auto', 'solver': 'lsqr'}"
...,...,...,...,...,...,...,...,...,...
196,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",88.22%,82.01%,96.90%,96.48%,82.01%,88.82%,96.31%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"
197,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",88.22%,82.01%,96.90%,96.48%,82.01%,88.82%,96.26%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"
198,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",88.22%,82.01%,96.90%,96.48%,82.01%,88.82%,96.25%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"
199,"[WDR11-DT, MAGOHB, ZNF542P, IFITM3, HSD17B3, N...",88.77%,82.97%,96.94%,96.48%,82.97%,89.40%,96.22%,"{'shrinkage': 0.5, 'solver': 'lsqr'}"


In [146]:
# df_result.to_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\sfs\SVM_DEG_training.csv", index=False)

In [147]:
xxx

NameError: name 'xxx' is not defined

# Feature select sfs

In [ ]:
df_result = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_on_training_27_5_AUC.csv", low_memory=False)

In [ ]:
df_result2 = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_DEG_on_training(test_selected)_27_5_AUC.csv", low_memory=False)

In [148]:
df_result = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_on_training_6_6.csv", low_memory=False)
df_result2 = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\LDA_DEG_on_training(test_selected)_6_6.csv", low_memory=False)

In [ ]:
# percent_cols = ["ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC"]

# for col in percent_cols:
#     df_result[col] = df_result[col].str.rstrip("%").astype(float)

In [149]:
for col in ["ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC"]:
    df_result[col] = df_result[col].str.rstrip("%").astype(float)
for col in ["ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC"]:
    df_result2[col] = df_result2[col].str.rstrip("%").astype(float)
# 1. Apply the Standard Definition of Balanced Accuracy
df_result['BA'] = (df_result['Sn'] + df_result['Sp']) / 2
# 1. Apply the Standard Definition of Balanced Accuracy
df_result2['BA'] = (df_result2['Sn'] + df_result['Sp']) / 2

In [ ]:
# Use '&' to ensure BOTH conditions are met
sp_threshold = 0
auc_threshold = 80.0

# This creates a strict filter
mask = (df_result["Sp"] >= sp_threshold) & (df_result["AUC"] >= auc_threshold)
filtered_df = df_result[mask]

if not filtered_df.empty:
    best_row = filtered_df.loc[filtered_df["BA"].idxmax()]
    print("Best Model Meeting Strict Criteria:")
    print(best_row)
else:
    print(f"No models found with Sp > {sp_threshold} and AUC > {auc_threshold}.")
    # Optional: see what the highest Sp available is
    print(f"Max Sp in data: {df_result['Sp'].max()}")

Best Model Meeting Strict Criteria:
Gene         ['WDR11-DT', 'MAGOHB', 'ZNF542P', 'IFITM3', 'H...
ACC                                                      88.08
Recall                                                   81.77
Precision                                                96.89
Sp                                                       96.48
Sn                                                       81.77
f1_Score                                                 88.68
AUC                                                      96.63
Parameter                 {'shrinkage': 0.5, 'solver': 'lsqr'}
BA                                                      89.125
Name: 108, dtype: object


In [173]:
df_result2.loc[108]

Gene         ['WDR11-DT', 'MAGOHB', 'ZNF542P', 'IFITM3', 'H...
ACC                                                      87.98
Recall                                                   86.67
Precision                                                91.92
Sp                                                       89.74
Sn                                                       86.67
f1_Score                                                  89.2
AUC                                                      94.79
Parameter                 {'shrinkage': 0.5, 'solver': 'lsqr'}
BA                                                      91.575
Name: 108, dtype: object

In [ ]:
df_result2.loc[110]

Gene         ['WDR11-DT', 'MAGOHB', 'ZNF542P', 'IFITM3', 'H...
ACC                                                      86.89
Recall                                                   85.71
Precision                                                90.91
Sp                                                       88.46
Sn                                                       85.71
f1_Score                                                  88.2
AUC                                                      94.74
Parameter                 {'shrinkage': 0.5, 'solver': 'lsqr'}
BA                                                      90.615
Name: 110, dtype: object

In [ ]:
best_genes = df_result.loc[df_result["ACC"].idxmax(), "Gene"]

In [ ]:
# import ast

# # Convert string → list
# best_genes = ast.literal_eval(best_genes)

In [ ]:
# Select features
X_train = X_train[best_genes]
X_test  = X_test[best_genes]

In [ ]:
X_train

,ENSG00000240225,ENSG00000111196,ENSG00000227165,ENSG00000005961,ENSG00000142089,ENSG00000103512,ENSG00000130948,ENSG00000213465,ENSG00000103316,ENSG00000142856,...,ENSG00000105369,ENSG00000156304,ENSG00000128731,ENSG00000241973,ENSG00000178950,ENSG00000135899,ENSG00000173575,ENSG00000124383,ENSG00000007202,ENSG00000164808
0,5.029751,5.313933,6.830194,12.739048,5.163708,5.847213,3.421733,4.456664,6.060632,6.396570,...,2.977562,2.977562,4.880156,4.880156,6.010254,3.421733,6.667921,2.977562,5.064561,2.977562
1,4.937442,4.567253,5.657219,12.818652,6.938392,5.554333,2.977562,4.826059,5.273281,7.267184,...,2.977562,2.977562,4.567253,5.527353,4.894071,3.906867,6.391238,2.977562,4.595823,3.716893
2,6.350295,6.504281,7.332512,12.668418,5.189230,7.662607,4.741644,4.782346,5.001946,8.019397,...,4.009663,4.161411,4.967904,5.930801,5.098964,5.001946,6.435281,3.826072,4.292595,4.088622
3,7.587544,7.099868,7.701091,12.174109,6.064303,7.014400,5.640744,4.189897,5.880896,6.875797,...,3.690989,2.977562,4.709205,5.831012,5.155216,4.927056,6.595267,4.364921,4.189897,2.977562
4,2.977562,2.977562,6.625362,12.355583,6.279014,5.397404,4.777392,5.124102,5.989692,6.404094,...,4.288787,4.288787,5.397404,4.288787,5.989692,7.566784,8.893093,5.397404,5.397404,5.624376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1610,6.827255,6.566049,7.278084,13.250418,4.750060,6.910702,4.727932,4.267806,3.952941,8.081787,...,4.083676,4.332976,4.951880,7.161996,5.644966,5.092907,6.408566,3.853177,4.771792,4.423809
1611,5.055621,5.438894,5.789434,11.495049,6.730757,7.121808,4.181500,5.298584,2.977562,7.998309,...,5.055621,4.963363,5.981778,5.936123,5.370572,8.508766,8.062493,6.649091,5.736876,5.503925
1612,5.383968,5.934437,7.212175,12.435184,4.671672,7.225306,5.211901,4.348262,4.044359,7.638879,...,4.854550,4.521444,5.136278,6.414964,5.856632,5.035127,7.044439,3.991813,4.553085,4.854550
1613,2.977562,2.977562,6.291087,12.095441,6.916693,5.526078,2.977562,4.566351,5.526078,5.830206,...,2.977562,2.977562,5.830206,5.133454,4.566351,7.755323,8.549032,6.079483,2.977562,2.977562


# Test set

In [156]:
X_train = X_train.iloc[:,:119]

X_test = X_test.iloc[:,:119]

In [157]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [158]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [159]:
X_test

,WDR11-DT,MAGOHB,ZNF542P,IFITM3,HSD17B3,NOMO1,ITGB3BP,LINC01088,COL6A3,ENSG00000260661,...,BAG6,PPT1,ZNF667-AS1,SUCLG2,SYNE2,GYS1,MMP1,TMEM70,DLAT,DEFA3
0,6.451547,5.348242,5.063048,7.340270,4.175718,5.665411,8.002006,4.625584,5.027229,3.306779,...,5.027229,4.466006,3.168105,3.876974,6.616707,3.306779,4.721400,4.625584,3.770458,3.578456
1,6.630307,5.769564,5.769564,3.921878,4.450995,5.844078,6.824695,4.551511,5.373541,3.728479,...,4.644246,6.223364,4.081394,5.026358,7.686605,4.219024,4.340982,8.187028,4.219024,5.518569
2,4.806340,4.468396,2.830723,7.664287,3.783802,4.345067,4.579964,3.783802,6.068926,2.830723,...,6.080541,5.466132,2.830723,4.345067,7.251075,4.543922,4.946281,4.919485,3.387581,3.387581
3,6.948542,6.390590,5.743158,5.220952,3.531932,6.448757,7.818158,4.996651,4.023021,4.564307,...,4.443469,4.308590,3.439457,4.023021,4.535230,4.272186,4.774914,8.362044,3.870284,2.830723
4,7.263197,6.852681,6.983045,5.905861,4.972501,6.667009,7.623942,4.838434,3.564025,4.543893,...,4.075577,4.150921,3.564025,2.830723,4.518003,3.564025,4.318690,6.917097,3.632339,3.297474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,7.793642,7.068577,6.270715,4.704627,5.540731,6.599390,9.150536,4.909011,5.282180,4.881746,...,4.605662,2.830723,4.336929,3.384360,2.830723,2.830723,3.981623,7.151849,2.830723,3.609025
179,6.029134,4.743265,5.561667,4.795813,3.522366,4.875604,7.151842,3.920887,5.369384,3.897971,...,4.383948,2.830723,4.007310,3.348412,2.830723,3.897971,4.665779,7.355216,2.830723,3.483490
180,6.585824,4.597360,5.647809,6.105820,4.280496,6.059986,8.286562,4.396034,6.193255,4.339747,...,4.921612,5.493568,4.280496,3.920949,5.963589,4.080345,4.339747,4.686642,2.830723,2.830723
181,4.978482,4.240479,4.631622,7.745341,2.830723,4.926833,6.610647,3.773898,5.122217,3.605274,...,5.407381,2.830723,4.240479,3.913699,5.611744,2.830723,4.631622,3.605274,2.830723,4.142769


In [160]:
from sklearn.preprocessing import StandardScaler

gene_names_selected = X_train.columns.tolist()
scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [161]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [162]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, confusion_matrix, accuracy_score, roc_curve

# --- 1. Custom Scorers (Same as previous models) ---
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp)
    return 0.0

# Safe definition for Windows Multiprocessing
specificity_scorer = make_scorer(recall_score, pos_label=0)

def balanced_accuracy_medical(y_true, y_pred):
    sn = recall_score(y_true, y_pred, zero_division=0)
    sp = specificity_score(y_true, y_pred)
    return (sn + sp) / 2

scoring_metrics = {
    'ACC': 'accuracy',
    'Recall': 'recall',
    'Precision': 'precision',
    'Sp': specificity_scorer,
    'Sn': 'recall',
    'f1_Score': 'f1',
    'AUC': 'roc_auc',
    'BA': make_scorer(balanced_accuracy_medical) 
}

model = LinearDiscriminantAnalysis(priors=[0.5, 0.5])

# --- Explain the Parameters ---
# LDA throws errors if you mix certain solvers with shrinkage.
# Using a list of dictionaries ensures GridSearch only tests valid combinations.
params = [
    # 'svd' is the default solver. It doesn't use shrinkage. Good for data with many features.
    # {'solver': ['svd']},
    # 'lsqr' and 'eigen' can use 'shrinkage', which acts like regularization (similar to alpha/C).
    # This helps immensely if your overlapping genes in PCA are causing multicollinearity.
    {'solver': ['lsqr'], 'shrinkage': [0.5]}
]

grid_search = GridSearchCV(
    estimator=model, 
    param_grid=params, 
    cv=5, 
    verbose=1, 
    scoring=scoring_metrics,
    refit='BA', # Selects best solver based on Balanced Accuracy
    return_train_score=True, 
    n_jobs=-1
)

In [163]:
import time
start_time = time.time()
grid_search.fit(X_train, y_train['cancer_type'])
end_time = time.time()

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [174]:
import pandas as pd
import numpy as np

results = grid_search.cv_results_
df_results = pd.DataFrame(results)

# Match exactly your scoring dict keys
metrics = ['ACC', 'Recall', 'Precision', 'Sp', 'Sn', 'f1_Score', 'AUC', 'BA']

# Compute variance for all metrics
for m in metrics:
    df_results[f'var_test_{m}'] = df_results[f'std_test_{m}'] ** 2

# Build summary table
cols = ['params']
for m in metrics:
    cols += [f'mean_test_{m}', f'std_test_{m}', f'var_test_{m}']
cols.append('rank_test_BA')

summary_table = df_results[cols].sort_values('rank_test_BA')

# Optional: scale to percentage and round
for m in metrics:
    summary_table[f'mean_test_{m}'] = (summary_table[f'mean_test_{m}'] * 100).round(4)
    summary_table[f'std_test_{m}']  = (summary_table[f'std_test_{m}']  * 100).round(4)
    # summary_table[f'var_test_{m}']  = (summary_table[f'var_test_{m}']  * 100).round(4)

print(summary_table)

                                 params  mean_test_ACC  std_test_ACC  \
0  {'shrinkage': 0.5, 'solver': 'lsqr'}        88.0822        0.5479   

   var_test_ACC  mean_test_Recall  std_test_Recall  var_test_Recall  \
0       0.00003           81.7699           0.9799         0.000096   

   mean_test_Precision  std_test_Precision  var_test_Precision  ...  \
0              96.8922              1.0201            0.000104  ...   

   mean_test_f1_Score  std_test_f1_Score  var_test_f1_Score  mean_test_AUC  \
0             88.6833             0.5641           0.000032        96.6251   

   std_test_AUC  var_test_AUC  mean_test_BA  std_test_BA  var_test_BA  \
0        0.4873      0.000024       89.1261        0.583     0.000034   

   rank_test_BA  
0             1  

[1 rows x 26 columns]


In [175]:
summary_table.head(10)    

,params,mean_test_ACC,std_test_ACC,var_test_ACC,mean_test_Recall,std_test_Recall,var_test_Recall,mean_test_Precision,std_test_Precision,var_test_Precision,...,mean_test_f1_Score,std_test_f1_Score,var_test_f1_Score,mean_test_AUC,std_test_AUC,var_test_AUC,mean_test_BA,std_test_BA,var_test_BA,rank_test_BA
0,"{'shrinkage': 0.5, 'solver': 'lsqr'}",88.0822,0.5479,0.00003,81.7699,0.9799,0.000096,96.8922,1.0201,0.000104,...,88.6833,0.5641,0.000032,96.6251,0.4873,0.000024,89.1261,0.583,0.000034,1


In [166]:
# Calculate training time
training_time = end_time - start_time

In [167]:
y_pred = grid_search.predict(X_test)

In [168]:
from sklearn.metrics import classification_report

### Reslut for testing set

In [169]:
best_idx = grid_search.best_index_
cv_res = grid_search.cv_results_
best_model = grid_search.best_estimator_

def fmt(metric):
    return f"{cv_res[f'mean_test_{metric}'][best_idx] * 100:.2f}%"

# --- THRESHOLD LOCK (Crucial for MLP Imbalance) ---
y_prob_train = best_model.predict_proba(X_train)[:, 1]
fpr_tr, tpr_tr, thresholds_tr = roc_curve(y_train, y_prob_train)
best_threshold = thresholds_tr[np.argmax(tpr_tr - fpr_tr)]
y_prob_test = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob_test >= best_threshold).astype(int)

In [170]:
from sklearn.metrics import roc_curve, auc

In [171]:
print("Training time:", training_time, "seconds")
print(classification_report( y_test['cancer_type'],y_pred,digits=6))


Training time: 4.4362242221832275 seconds
              precision    recall  f1-score   support

           0   0.833333  0.897436  0.864198        78
           1   0.919192  0.866667  0.892157       105

    accuracy                       0.879781       183
   macro avg   0.876263  0.882051  0.878177       183
weighted avg   0.882596  0.879781  0.880240       183

